In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# パフォーマンス管理: Q-CTRL Fire Opal による Qiskit Function
*[API リファレンス](https://docs.quantum.ibm.com/api/functions/q-ctrl-performance-management)をご覧ください*

> **Note:** Qiskit Functions は実験的な機能であり、IBM Quantum&reg; Premium Plan、Flex Plan、および On-Prem（IBM Quantum Platform API 経由）Plan のユーザーのみが利用できます。現在プレビュー版として提供されており、変更される可能性があります。


<Accordion>
<AccordionItem title="Package versions">

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.3.1
qiskit-ibm-runtime~=0.45.1
```
</AccordionItem>
</Accordion>
## 概要
Fire Opal Performance Management を使用すると、量子ハードウェアの専門家でなくても、大規模な量子コンピューターから有意義な結果を簡単に得ることができます。Fire Opal Performance Management でCircuitを実行すると、AI を活用したエラー抑制技術が自動的に適用され、より多くのGateとQubitを使用した大規模な問題のスケーリングが可能になります。このアプローチにより、正解に到達するために必要なショット数が削減され、追加のオーバーヘッドも発生しません。その結果、計算時間とコストの両方で大幅な節約が実現します。

Performance Management はエラーを抑制し、ノイズの多いハードウェアで正しい答えが得られる確率を高めます。言い換えると、信号対雑音比を向上させます。次の画像は、Performance Management によって実現された精度向上により、10 qubit の量子フーリエ変換アルゴリズムの場合に追加ショットの必要性がどれほど削減されるかを示しています。わずか 30 ショットで、Q-CTRL は 99% の信頼度閾値に到達します。一方、デフォルト設定（`QiskitRuntime` Sampler、`optimization_level`=3、`resilience_level`=1、`ibm_sherbrooke`）では 170,000 ショットが必要です。正しい答えをより早く得ることで、計算ランタイムを大幅に節約できます。

![改善されたランタイムの可視化](../docs/images/guides/qctrl-performance-management/achieve_more.svg)

Performance Management 関数はあらゆるアルゴリズムに使用でき、標準的な [Qiskit Runtime プリミティブ](/guides/primitives)の代わりに簡単に利用できます。内部では、複数のエラー抑制技術が連携して実行時のエラー発生を防ぎます。Fire Opal のパイプラインメソッドはすべて事前設定済みでアルゴリズムに依存しないため、常に最高のパフォーマンスをすぐに発揮できます。

Performance Management へのアクセスを取得するには、[Q-CTRL にお問い合わせ](https://form.typeform.com/to/uOAVDnGg?typeform-source=q-ctrl.com)ください。
## 説明
Fire Opal Performance Management には、Qiskit Runtime プリミティブと同様の 2 つの実行オプションがあり、Q-CTRL の Sampler と Estimator に簡単に切り替えることができます。Performance Management 関数を使用する際の一般的なワークフローは次のとおりです。
1. Circuit を定義する（Estimator の場合はオペレーターも定義する）。
2. Circuit を実行する。
3. 結果を取得する。

ハードウェアのノイズを低減するために、Fire Opal は以下の画像に示すような AI を活用したエラー抑制技術を幅広く採用しています。Fire Opal では、パイプライン全体が完全に自動化されており、設定は一切不要です。

Fire Opal のパイプラインは、量子ランタイムの増加や余分な物理 qubit など、追加のオーバーヘッドの必要性を排除します。ただし、古典的な処理時間は依然として考慮すべき要素です（推定値については [ベンチマーク](#benchmarks) セクションを参照してください。「Total time」は古典的および量子的な処理の両方を反映しています）。サンプリングの形でオーバーヘッドを必要とするエラー軽減とは対照的に、Fire Opal のエラー抑制は Gate レベルとパルスレベルの両方で機能し、さまざまなノイズの発生源に対処してエラーが発生する可能性を防ぎます。エラーを防ぐことで、コストのかかる後処理の必要性が排除されます。

次の画像は、Fire Opal Performance Management によって自動化されるエラー抑制手法を示しています。

![エラー抑制パイプラインの可視化](../docs/images/guides/qctrl-performance-management/error_suppression.svg)

この関数は Sampler と Estimator の 2 つのプリミティブを提供しており、両方の入力と出力は [Qiskit Runtime V2 プリミティブ](/guides/primitive-input-output#pubs)の実装仕様を拡張しています。
## ベンチマーク
[公開されたアルゴリズムベンチマーク](https://journals.aps.org/prapplied/abstract/10.1103/PhysRevApplied.20.024034)の結果は、Bernstein-Vazirani、量子フーリエ変換、Grover の探索、量子近似最適化アルゴリズム、変分量子固有値ソルバーなど、さまざまなアルゴリズムにわたる大幅な性能向上を示しています。このセクションの残りの部分では、実行できるアルゴリズムの種類と、期待されるパフォーマンスおよびランタイムについて詳しく説明します。

以下の独立した研究は、Q-CTRL の Performance Management が記録的な規模でのアルゴリズム研究を可能にすることを示しています：
- [Parametrized Energy-Efficient Quantum Kernels for Network Service Fault Diagnosis](https://arxiv.org/abs/2405.09724v1) — 最大 50 qubit の量子カーネル学習
- [Tensor-based quantum phase difference estimation for large-scale demonstration](https://arxiv.org/abs/2408.04946) — 最大 33 qubit の量子位相推定
- [Hierarchical Learning for Quantum ML: Novel Training Technique for Large-Scale Variational Quantum Circuits](https://arxiv.org/abs/2311.12929) — 最大 21 qubit の量子データローディング

以下の表は、`ibm_fez` での過去のベンチマーク実行から得られた精度とランタイムの概算ガイドを提供しています。他のデバイスでのパフォーマンスは異なる場合があります。使用時間は Circuit あたり 10,000 ショットを前提としています。「qubit 数」はハードの制限ではなく、非常に一貫したソリューション精度が期待できるおおよその閾値を表しています。より大きな問題サイズも正常に解決されており、これらの限界を超えたテストも推奨されています。

| 例    | qubit 数 | 精度 | 精度の指標 | 総時間 (秒) | ランタイム使用量 (秒) | プリミティブ（モード） |
| ---------  | ---------------- | -------------------------- | -------- | ---------- | ------------- |------------- |
| Bernstein–Vazirani  |  50Q    | 100%  | 成功率（正しい答えが最も多いビット文字列となった実行の割合）     | 10    | 8         | Sampler |
| 量子フーリエ変換 | 30Q              | 100% | 成功率（正しい答えが最も多いビット文字列となった実行の割合）      | 10    | 8        | Sampler |
| 量子位相推定  | 30Q   | 99.9998%  | 求められた角度の精度: `1- abs(real_angle - angle_found)/pi`  | 10  | 8  | Sampler |
| 量子シミュレーション: Ising モデル（15 ステップ） | 20Q  | 99.775%   |  $A$（以下で定義）  |  60（ステップあたり）  | 15（ステップあたり）   | Estimator |
| 量子シミュレーション 2: 分子動力学（20 タイムポイント） | 34Q  |  96.78%  |  $A_{mean}$（以下で定義）   | 10（タイムポイントあたり）    | 6（タイムポイントあたり）  | Estimator |

期待値の測定精度の定義 — 指標 $A$ は次のように定義されます：
$$
A = 1 - \frac{|\epsilon^{ideal} - \epsilon^{meas}|}{\epsilon^{ideal}_{max} - \epsilon^{ideal}_{min}},
$$
ここで、$ \epsilon^{ideal} $ = 理想的な期待値、$ \epsilon^{meas} $ = 測定された期待値、$\epsilon^{ideal}_{max} $ = 理想的な最大値、$\epsilon^{ideal}_{min}$ = 理想的な最小値です。$A_{mean}$ は単純に複数の測定にわたる $A$ の値の平均です。

この指標は、得られる値の範囲におけるグローバルなシフトやスケーリングに対して不変であるため使用されています。言い換えると、可能な期待値の範囲を上下にシフトしたり広げたりしても、$A$ の値は一定のままであるべきです。
## 始め方
Fire Opal Performance Management は推奨バージョンである Qiskit v`2.0.0` を使用します。サポートされているバージョンは Qiskit >=v`2.0.0` です。
[IBM Quantum Platform API キー](http://quantum.cloud.ibm.com/)を使用して認証し、以下のように Qiskit Function を選択します。（このスニペットは、すでに[アカウントをローカル環境に保存](/guides/functions#install-qiskit-functions-catalog-client)していることを前提としています。）

In [1]:
# This cell is hidden from users. It hides the "...You have imported samplomatic..." warning.
import warnings

warnings.filterwarnings("ignore", message=".*You have imported samplomatic*")

In [2]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [3]:
# Access Function
perf_mgmt = catalog.load("q-ctrl/performance-management")

<Admonition type="note" title="Does this function support all IBM backends?">
If you want to use a backend that this function does not currently support, [reach out to Q-CTRL](https://form.typeform.com/to/iuujEAEI?typeform-source=q-ctrl.com) to add support.
</Admonition>

## Estimator primitive

### Estimator example

Use Fire Opal Performance Management's Estimator primitive to determine the expectation value of a single circuit-observable pair.

In addition to the `qiskit-ibm-catalog` and `qiskit` packages, you will also use the `numpy` package to run this example. You can install this package by uncommenting the following cell if you are running this example in a notebook using the IPython kernel.

In [4]:
# %pip install numpy

**2. Circuit を実行する**

Circuit を実行し、必要に応じてバックエンドとショット数を定義します。

In [5]:
import numpy as np
from qiskit.circuit.library import iqp
from qiskit.quantum_info import random_hermitian, SparsePauliOp

n_qubits = 50

# Generate a random circuit
mat = np.real(random_hermitian(n_qubits, seed=1234))
circuit = iqp(mat)
circuit.measure_all()

# Define observables as a string
observable = SparsePauliOp("Z" * n_qubits)

In [6]:
# Create PUB tuple
estimator_pubs = [(circuit, observable)]

おなじみの [Qiskit Serverless API](/guides/serverless) を使用して、Qiskit Function ワークロードのステータスを確認できます：

In [7]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy().name

In [8]:
# Run the circuit using Estimator
qctrl_estimator_job = perf_mgmt.run(
    primitive="estimator",
    pubs=estimator_pubs,
    backend_name=backend_name,
)

**3. 結果を取得する**

In [9]:
qctrl_estimator_job.status()

'QUEUED'

結果は [Estimator の結果](/guides/estimator-input-output)と同じ形式です：

In [10]:
# Retrieve the counts from the result list
result = qctrl_estimator_job.result()

The results have the same format as an [Estimator result](/docs/guides/estimator-input-output):

In [22]:
import numpy

result_str = str(result)

with numpy.printoptions(threshold=200):
    print(
        f"The result of the submitted job had {len(result)} PUB "
        f"and has a value:\n {result[0]}\n"
    )

print("The associated PubResult of this job has the following DataBins:")
print(f"{result[0].data}\n")

print(f"And this DataBin has attributes: {result[0].data.keys()}")

print("The expectation values measured from this PUB are:")
print(f"{result[0].data.evs}")

The result of the submitted job had 1 PUB
The result of the submitted job had 1 PUB and has a value:
 PubResult(data=DataBin(evs=0.0195, stds=0.9998098569228051), metadata={'precision': None})

The associated PubResult of this job has the following DataBins:
DataBin(evs=0.0195, stds=0.9998098569228051)

And this DataBin has attributes: dict_keys(['evs', 'stds'])
The expectation values measured from this PUB are:
0.0195


## Sampler プリミティブ
### Sampler の使用例
Fire Opal Performance Management の Sampler プリミティブを使用して、Bernstein–Vazirani 回路を実行します。このアルゴリズムはブラックボックス関数の出力から隠れた文字列を見つけるもので、正解が一意に定まるため、一般的なベンチマークアルゴリズムとしてよく使われています。

**1. 回路を作成する**

アルゴリズムの正解である隠れビット列と Bernstein–Vazirani 回路を定義します。`circuit_width` を変更するだけで回路の幅を調整できます。

In [12]:
import qiskit

circuit_width = 35
hidden_bitstring = "1" * circuit_width

# Create circuit, reserving one qubit for BV oracle
bv_circuit = qiskit.QuantumCircuit(circuit_width + 1, circuit_width)
bv_circuit.x(circuit_width)
bv_circuit.h(range(circuit_width + 1))
for input_qubit, bit in enumerate(reversed(hidden_bitstring)):
    if bit == "1":
        bv_circuit.cx(input_qubit, circuit_width)
bv_circuit.barrier()
bv_circuit.h(range(circuit_width + 1))
bv_circuit.barrier()
for input_qubit in range(circuit_width):
    bv_circuit.measure(input_qubit, input_qubit)

# Create PUB tuple
sampler_pubs = [(bv_circuit,)]

**2. 回路を実行する**

回路を実行します。必要に応じて Backend とショット数を指定できます。

In [13]:
# Run the circuit using Sampler
qctrl_sampler_job = perf_mgmt.run(
    primitive="sampler",
    pubs=sampler_pubs,
    backend_name=backend_name,
)

Qiskit Function ワークロードの[ステータスを確認する](/guides/functions#check-job-status)か、以下のように[結果を取得する](/guides/functions#retrieve-results)ことができます。

In [14]:
# Print the ID so you can use it later, if necessary
print(qctrl_sampler_job.job_id)

qctrl_sampler_job.status()

60fe2fa1-a860-43e4-8615-c6ac4180f93b


'QUEUED'

**3. Retrieve the result**

In [15]:
# Retrieve the job results
sampler_result = qctrl_sampler_job.result()

In [16]:
# Get results for the first (and only) PUB
pub_result = sampler_result[0]
counts = pub_result.data.c.get_counts()

print("Counts for the meas output register (limited to 30 results):")
for i, (bitstring, count) in enumerate(counts.items()):
    if i >= 50:
        print(f"  ... ({len(counts) - 30} more items)")
        break
    print(f"  {bitstring}: {count}")

Counts for the meas output register (limited to 30 results):
  11111111111111111111111111111111111: 1661
  11111111111111111111111111110111111: 60
  11111111111111111111111111111101111: 54
  11111111111111111111111111111110111: 54
  11111111111111011111111111111111111: 46
  11111111111111111110111111111111111: 44
  11111111111111111111111101111111111: 42
  11111111111111111111111110111111111: 42
  11111111111111110111111111111111111: 41
  11111111111111111111111111111111101: 39
  11111111111111111111101111111111111: 38
  11111111111111111111110111111111111: 38
  11111111111111111111111111101111111: 37
  11111111111111111111111111111111110: 36
  11111111111110111111111111111111111: 35
  11111111111111111111111111111011111: 32
  11111111111111101111111111111111111: 32
  01111111111111111111111111111111111: 27
  11111111111111111011111111111111111: 23
  11111111101111111111111111111111111: 22
  11111111111111111111111111111111011: 21
  11111111011111111111111111111111111: 20
  00000000000

**3. Plot the top bitstrings**

Plot the bitstring with the highest counts to see if the hidden bitstring was the mode.

In [17]:
import matplotlib.pyplot as plt


def plot_top_bitstrings(counts_dict, hidden_bitstring=None):
    # Sort and take the top 100 bitstrings
    top_100 = sorted(counts_dict.items(), key=lambda x: x[1], reverse=True)[
        :100
    ]
    if not top_100:
        print("No bitstrings found in the input dictionary.")
        return

    # Unzip the bitstrings and their counts
    bitstrings, counts = zip(*top_100)

    # Assign colors: purple if the bitstring matches hidden_bitstring,
    # otherwise gray
    colors = [
        "#680CE9" if bit == hidden_bitstring else "gray" for bit in bitstrings
    ]

    # Create the bar plot
    plt.figure(figsize=(15, 8))
    plt.bar(
        range(len(bitstrings)), counts, tick_label=bitstrings, color=colors
    )

    # Rotate the bitstrings for better readability
    plt.xticks(rotation=90, fontsize=8)
    plt.xlabel("Bitstrings")
    plt.ylabel("Counts")
    plt.title("Top 100 Bitstrings by Counts")

    # Show the plot
    plt.tight_layout()
    plt.show()

The hidden bitstring is highlighted in purple, and it should be the bitstring with the highest number of counts.

In [18]:
plot_top_bitstrings(counts, hidden_bitstring)

<Image src="../docs/images/guides/q-ctrl-performance-management/extracted-outputs/8106d906-0.avif" alt="Output of the previous code cell" />

**3. 上位ビット列をプロットする**

最も出現回数の多いビット列をプロットして、隠れビット列が最頻値になっているかどうかを確認します。